# Modelling Jamaica's infrastructure river flood exposure and risk

## Step 0: Import relevant things and set up base directory

#### Step 0.1: Check which python environment I am in. Change the kernal (top right) to snail_env so that snail works.

In [ ]:
!which python

#### Step 0.2: Import things

In [ ]:
# Imports from Python standard library
import os
import pyproj


from pathlib import Path

# see https://docs.python.org/3/library/glob.html
from glob import glob

# Imports from other Python packages
import geopandas as gpd
gpd._compat.USE_PYGEOS = False

# numpy is used by pandas and geopandas to store data in efficient arrays
# we use it in this notebook to help with trapezoidal integration
# see https://numpy.org/
import numpy as np
import pandas as pd


# seaborn helps produce more complex plots
# see https://seaborn.pydata.org/
import seaborn as sns

from scipy.integrate import simpson

import snail.damages
import snail.intersection
import snail.io

from pyproj import Geod

# tqdm lets us show progress bars (and تقدّم means "progress" in Arabic)
# see https://tqdm.github.io/
from tqdm.notebook import tqdm


In [ ]:
# # Update this path if needed: usually it is within your snail_env directory.
# os.environ["PROJ_LIB"] = "/opt/miniconda3/envs/snail_env/share/proj"
# print("PROJ data directory set to:", pyproj.datadir.get_data_dir())

In [ ]:
print(os.path.exists("/opt/miniconda3/envs/snail_env/share/proj"))

#### Step 0.3: Set up base paths

In [ ]:
# dir = (
#     Path(os.getcwd()).resolve().parents[3]
# )  # get parent directory of snail package
# data_folder = os.path.join(dir, "ghana_tutorial")
# # data_folder = Path("YOUR_PATH/ghana_tutorial")

# Define the base path

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")

# Define the folder for hazard files relative to base_path
hazards_path = base_path / "Processed_data/Hazards"

# Define the folder for infrastructure files relative to base_path
networks_folder = base_path / "Processed_data/networks"


# Define the folder for infrastructure files relative to base_path
infrastructure_folder = base_path / "Processed_data/Infrastructure"



In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"
pyproj.datadir.get_data_dir()

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
# print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

## Step 1: Analyse exposure

In [ ]:
# List all files in the directory to check that it is pointing to the right place
all_files = list(hazards_path.iterdir())
display("All files in directory:", [f.name for f in all_files])

In [ ]:
# Read in river flood maps: found via processed data -> hazards -> Global Flood Map -> Jamaica -> Fluvial -> Raw Depths
# Tifs as follows: [Q20_RD_02.tif; Q50_RD_02.tif; Q100_RD_02.tif; Q200_RD_02.tif; Q500_RD_02.tif]  

river_flood_tiff_files = list(hazards_path.glob('*.tif'))

In [ ]:
# Convert to strings if required by downstream functions
hazards_path = [str(path) for path in hazards_path]  # Only needed if strings are required

# Create a DataFrame
hazard_files = pd.DataFrame({"path": hazards_path}) 

# First time around, with snapped points - to find DEM grid index
# hazard_files = pd.DataFrame({"path": ["path/to/dem.tif"]})

# Second time around, with pre-snapped points - to find flood raster grid index
# hazard_files = pd.DataFrame({"path": ["path/to/rp1500.tif"]

# Extract file stems
hazard_files["key"] = [Path(path).stem for path in hazards_path]  # Path(path) ensures compatibility

# Extend raster metadata (check if it requires strings)
hazard_files, grids = snail.io.extend_rasters_metadata(hazard_files)

print(hazard_files.head(5))



#List all the hazard files in the flood_layer folder:

# hazard_paths = sorted(
#     glob(str(data_folder + "/flood_layer/gha/wri_aqueduct_version_2/wri*.tif"))
# )

# Specify the subdirectory and file pattern / find hazard paths
hazard_paths = sorted(hazard_folder.glob("*.tif"))

# Convert to strings if required by downstream functions
# hazard_paths = [str(path) for path in hazard_paths]  # Only needed if strings are required

# Create a DataFrame
hazard_files = pd.DataFrame({"path": hazard_paths})

# Extract file stems
hazard_files["key"] = [Path(path).stem for path in hazard_paths]  # Path(path) ensures compatibility

# Extend raster metadata (check if it requires strings)
hazard_files, grids = snail.io.extend_rasters_metadata(hazard_files)

# Display the first 5 rows
print(hazard_files.head(5))
#hazard_files.head(5)









In [ ]:
assert len(grids) == 1
grid = grids[0]

### Step 2.2 Read in roads again, then do intersections against all hazard scenarios.

In [ ]:
# roads_file = data_folder + "/GHA_OSM_roads.gpkg"
# roads = gpd.read_file(roads_file, layer="edges")
# roads.head(2)

# Path to your roads_edges GeoPackage
roads_gpkg_path = networks_folder / "transport/roads.gpkg"

# Load the roads_edges data
roads_gdf = gpd.read_file(roads_gpkg_path)
roads_gdf = roads_gdf.to_crs(jamaica_metric_grid_crs)

roads_nodes_gdf = gpd.read_file(roads_gpkg_path, layer="nodes")
roads_nodes_gdf = roads_nodes_gdf.to_crs(jamaica_metric_grid_crs)

node_count = len(roads_nodes_gdf)
print(node_count)


In [ ]:
# split roads along hazard data grid

# TODO top-level "overlay_rasters"
# TODO for vector in vectors / for raster in rasters "overlay_raster"


# push into split_linestrings, flag to disable
prepared = snail.intersection.prepare_linestrings(roads)

flood_intersections = snail.intersection.split_linestrings(prepared, grid)

# push into split_linestrings
flood_intersections = snail.intersection.apply_indices(
    flood_intersections, grid, index_i="i_0", index_j="j_0"  # index_i="dem_i",   # index_i="flood_i"
)

flood_intersections = snail.io.associate_raster_files(
    flood_intersections, hazard_files
)

# calculate the length of each stretch of road
# don't include in snail wrapper top-level function
geod = Geod(ellps="WGS84")
flood_intersections["length_m"] = flood_intersections.geometry.apply(
    geod.geometry_length
)



In [ ]:
# save to file
output_file = os.path.join(
    data_folder,
    "results",
    str(Path(roads_file).name).replace(
        ".gpkg", "_edges___exposure.geoparquet"
    ),
)

flood_intersections.to_parquet(output_file)

In [ ]:
flood_intersections.columns

In [ ]:
data_cols = [col for col in flood_intersections.columns if "wri" in col]

In [ ]:
data_cols

In [ ]:
# find any max depth and filter > 0
all_intersections = flood_intersections[
    flood_intersections[data_cols].max(axis=1) > 0
]
# subset columns
all_intersections = all_intersections.drop(
    columns=["osm_id", "name", "from_id", "to_id", "geometry", "i_0", "j_0"]
)
# melt and check again for depth
all_intersections = all_intersections.melt(
    id_vars=["id", "split", "road_type", "length_m"],
    value_vars=data_cols,
    var_name="key",
    value_name="depth_m",
).query("depth_m > 0")

all_intersections.head(5)

In [ ]:
river = all_intersections[all_intersections.key.str.contains("inunriver")]
#coast = all_intersections[all_intersections.key.str.contains("inuncoast")]

# coast_keys = coast.key.str.extract(
#     r"wri_aqueduct-version_2-(?P<hazard>\w+)_(?P<rcp>[^_]+)_(?P<sub>[^_]+)_(?P<epoch>[^_]+)_rp(?P<rp>[^-]+)-gha"
# )
# coast = pd.concat([coast, coast_keys], axis=1)
river_keys = river.key.str.extract(
    r"wri_aqueduct-version_2-(?P<hazard>\w+)_(?P<rcp>[^_]+)_(?P<gcm>[^_]+)_(?P<epoch>[^_]+)_rp(?P<rp>[^-]+)-gha"
)
river = pd.concat([river, river_keys], axis=1)

In [ ]:
# river.rp = river.rp.apply(lambda rp: float(rp.replace("_", ".").lstrip("0")))
river.gcm = river.gcm.str.replace("0", "")
river.head(5)

In [ ]:
Summarise total length of roads exposed to depth 2m or greater river flooding, under different return periods and climate scenarios:

summary = (
    river[river.depth_m >= 2.0]
    .drop(columns=["id", "split", "road_type", "key"])
    .groupby(["hazard", "rcp", "gcm", "epoch", "rp"])
    .sum()
    .drop(columns=["depth_m"])
)

summary

In [ ]:
# Plot exposure against return period, with separate plot areas for each Representative Concentration Pathway (RCP), 
# and different colours for the different Global Climate Models (GCM):

plot_data = summary.reset_index()
plot_data = plot_data[plot_data.epoch.isin(["1980", "2080"])]
plot_data.rp = plot_data.rp.apply(lambda rp: int(rp.lstrip("0")))
plot_data["probability"] = 1 / plot_data.rp
plot_data.head(5)

In [ ]:
sns.relplot(
    data=plot_data,
    x="rp",
    y="length_m",
    hue="gcm",
    col="rcp",
    kind="line",
    marker="o",
)

## Step 2: Analyse vulnerability

In [ ]:
# Set up fragility curve assumptions, where probability of damage (pfail) depends on whether a road is paved and the depth of flood it is exposed to.

# These assumptions are derived from Koks, E.E., Rozenberg, J., Zorn, C. et al. A global multi-hazard risk analysis of road and railway infrastructure assets. Nat Commun 10, 2677 (2019). https://doi.org/10.1038/s41467-019-10442-3, Figure S3, extrapolated to 2m and 3m depths.

# The analysis is likely to be highly sensitive to these assumptions, and this approach is strongly limited by the availability and quality of fragility data, as well as the assumption that fragility can be related to flood depth alone - flood water velocity would be an important factor in a more detailed vulnerability assessment.



In [ ]:
paved = snail.damages.PiecewiseLinearDamageCurve(
    pd.DataFrame(
        {
            "intensity": [0.0, 0.999999999, 1, 2, 3],
            "damage": [0.0, 0.0, 0.1, 0.3, 0.5],
        }
    )
)
unpaved = snail.damages.PiecewiseLinearDamageCurve(
    pd.DataFrame(
        {
            "intensity": [0.0, 0.999999999, 1, 2, 3],
            "damage": [0.0, 0.0, 0.9, 1.0, 1.0],
        }
    )
)
paved, unpaved

In [ ]:
paved.plot(), unpaved.plot()

In [ ]:
# Set up cost assumptions.

# These are taken from Koks et al (2019) again, Table S8, construction costs to be assumed as an estimate of full rehabilitation after flood damage.

# Again the analysis is likely to be highly sensitive to these assumptions, which should be replaced by better estimates if available.

In [ ]:
costs = pd.DataFrame(
    {
        "kind": ["paved_four_lane", "paved_two_lane", "unpaved"],
        "cost_usd_per_km": [3_800_000, 932_740, 22_780],
    }
)
costs

In [ ]:
#Set up assumptions about which roads are paved or unpaved, and number of lanes.

sorted(river.road_type.unique())



In [ ]:
# Assume all tertiary roads are unpaved, all others are paved.

river["paved"] = ~(river.road_type == "tertiary")


In [ ]:
def kind(road_type):
    if road_type in ("trunk", "trunk_link", "motorway"):
        return "paved_four_lane"
    elif road_type in ("primary", "primary_link", "secondary"):
        return "paved_two_lane"
    else:
        return "unpaved"


river["kind"] = river.road_type.apply(kind)

In [ ]:
river = river.merge(costs, on="kind")


In [ ]:
Use the damage curve to estimate proportion_damaged for each exposed section.



In [ ]:
river.head(2)


In [ ]:
paved_depths = river.loc[river.paved, "depth_m"]
paved_damage = paved.damage_fraction(paved_depths)
river.loc[river.paved, "proportion_damaged"] = paved_damage

unpaved_depths = river.loc[~river.paved, "depth_m"]
unpaved_damage = paved.damage_fraction(unpaved_depths)
river.loc[~river.paved, "proportion_damaged"] = unpaved_damage

In [ ]:
# Finally estimate cost of rehabilitation for each exposed section

river["damage_usd"] = river.length_m * river.cost_usd_per_km * 1e-3
river.head(2)

In [ ]:
river.to_csv(
    os.path.join(data_folder, "results/inunriver_damages_rp.csv"), index=False
)

In [ ]:
summary = (
    river.drop(
        columns=[
            "id",
            "split",
            "length_m",
            "key",
            "depth_m",
            "paved",
            "kind",
            "cost_usd_per_km",
            "proportion_damaged",
        ]
    )
    .groupby(["road_type", "hazard", "rcp", "gcm", "epoch", "rp"])
    .sum()
)
summary

## Step 3: Analyse Risk¶


In [ ]:
# Calculate expected annual damages for each road under historical hazard.

# Start by selecting only historical intersections, and keeping only the road ID, return period, and cost of rehabilitation if damaged.


In [ ]:
historical = river[river.rcp == "historical"][["id", "rp", "damage_usd"]]


In [ ]:
# Sum up the expected damage for each road, per return period, then pivot the table to create columns for each return period - now there is one row per road.

In [ ]:
historical = historical.groupby(["id", "rp"]).sum().reset_index()
historical = historical.pivot(index="id", columns="rp").replace(
    float("NaN"), 0
)
historical.columns = [f"rp{int(rp)}" for _, rp in historical.columns]
historical.head(2)

In [ ]:
# Calculate expected annual damages, integrating under the expected damage curve over return periods.

def calculate_ead(df):
    rp_cols = sorted(
        list(df.columns), key=lambda col: 1 / int(col.replace("rp", ""))
    )
    rps = np.array([int(col.replace("rp", "")) for col in rp_cols])
    probabilities = 1 / rps
    rp_damages = df[rp_cols]
    return simpson(rp_damages, x=probabilities, axis=1)


historical["ead_usd"] = calculate_ead(historical)
historical.head(2)

historical.to_csv(
    os.path.join(data_folder, "results/inunriver_damages_ead__historical.csv")
)